<a href="https://colab.research.google.com/github/hishanthp2008-del/DAA/blob/main/DAA8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ---------------------------------------------------------
# TRAVELLING SALESMAN PROBLEM (TSP)
# Branch and Bound with Reduced Cost Matrix
# ---------------------------------------------------------

INF = float('inf')


# ---------------------------------------------------------
# COST MATRIX
# ---------------------------------------------------------
# Cities: 0, 1, 2, 3, 4
#
# cost[i][j] = cost of travelling from city i to city j
#
# INF means travelling from a city to itself is not allowed.

cost_matrix = [
    [INF, 20, 30, 10, 11],
    [15, INF, 16, 4, 2],
    [3, 5, INF, 2, 4],
    [19, 6, 18, INF, 3],
    [16, 4, 7, 16, INF]
]

N = len(cost_matrix)


# ---------------------------------------------------------
# REDUCE COST MATRIX
# ---------------------------------------------------------
# For every row and column, subtract the minimum finite
# value. The sum of all reductions gives the lower bound.

def reduce_matrix(matrix):

    n = len(matrix)
    reduction_cost = 0

    # Row reduction
    for i in range(n):

        row_values = [
            matrix[i][j]
            for j in range(n)
            if matrix[i][j] != INF
        ]

        if row_values:

            row_min = min(row_values)

            if row_min != 0:
                reduction_cost += row_min

                for j in range(n):
                    if matrix[i][j] != INF:
                        matrix[i][j] -= row_min

    # Column reduction
    for j in range(n):

        col_values = [
            matrix[i][j]
            for i in range(n)
            if matrix[i][j] != INF
        ]

        if col_values:

            col_min = min(col_values)

            if col_min != 0:
                reduction_cost += col_min

                for i in range(n):
                    if matrix[i][j] != INF:
                        matrix[i][j] -= col_min

    return reduction_cost


# ---------------------------------------------------------
# CREATE REDUCED MATRIX FOR A BRANCH
# ---------------------------------------------------------

def create_child_matrix(parent_matrix, from_city, to_city):

    matrix = [
        row.copy()
        for row in parent_matrix
    ]

    # No more outgoing edges from from_city
    for j in range(N):
        matrix[from_city][j] = INF

    # No more incoming edges to to_city
    for i in range(N):
        matrix[i][to_city] = INF

    # Prevent immediate return to the starting city
    # before completing the tour.
    matrix[to_city][0] = INF

    return matrix


# ---------------------------------------------------------
# BRANCH AND BOUND TSP
# ---------------------------------------------------------

def tsp_branch_and_bound():

    # Create a copy of the original cost matrix
    root_matrix = [
        row.copy()
        for row in cost_matrix
    ]

    # Calculate initial lower bound
    root_bound = reduce_matrix(root_matrix)

    # Store:
    # (lower_bound, current_city, path, matrix)
    priority_queue = [
        (root_bound, 0, [0], root_matrix)
    ]

    best_cost = INF
    best_path = None

    nodes_explored = 0

    while priority_queue:

        # Select node with smallest lower bound
        priority_queue.sort(key=lambda x: x[0])

        bound, current_city, path, matrix = priority_queue.pop(0)

        nodes_explored += 1

        # Prune if lower bound is already worse
        if bound >= best_cost:
            continue

        # If all cities have been visited
        if len(path) == N:

            # Cost of returning to starting city
            return_cost = cost_matrix[current_city][0]

            if return_cost != INF:

                total_cost = bound + return_cost

                if total_cost < best_cost:
                    best_cost = total_cost
                    best_path = path + [0]

            continue

        # Branch to every unvisited city
        for next_city in range(N):

            if next_city in path:
                continue

            edge_cost = cost_matrix[current_city][next_city]

            if edge_cost == INF:
                continue

            # Create child matrix
            child_matrix = create_child_matrix(
                matrix,
                current_city,
                next_city
            )

            # Lower bound of child
            child_bound = bound + edge_cost

            reduction = reduce_matrix(child_matrix)

            child_bound += reduction

            # Add child if it can still improve the solution
            if child_bound < best_cost:

                priority_queue.append(
                    (
                        child_bound,
                        next_city,
                        path + [next_city],
                        child_matrix
                    )
                )

    return best_path, best_cost, nodes_explored


# ---------------------------------------------------------
# DISPLAY COST MATRIX
# ---------------------------------------------------------

def print_cost_matrix():

    print("\nCost Matrix:")
    print("-" * 45)

    for row in cost_matrix:

        for value in row:

            if value == INF:
                print(f"{'INF':>8}", end="")
            else:
                print(f"{value:>8}", end="")

        print()

    print("-" * 45)


# ---------------------------------------------------------
# MAIN PROGRAM
# ---------------------------------------------------------

def main():

    print("=" * 60)
    print("TRAVELLING SALESMAN PROBLEM")
    print("BRANCH AND BOUND USING REDUCED COST MATRIX")
    print("=" * 60)

    print_cost_matrix()

    # Solve TSP
    optimal_path, minimum_cost, nodes_explored = (
        tsp_branch_and_bound()
    )

    print("\n" + "-" * 60)

    print("Optimal Tour:")
    print(" -> ".join(map(str, optimal_path)))

    print("\nMinimum Tour Cost:")
    print(minimum_cost)

    print("\nNodes Explored:")
    print(nodes_explored)

    print("-" * 60)

    # Verify the cost directly
    calculated_cost = 0

    for i in range(len(optimal_path) - 1):
        calculated_cost += cost_matrix[
            optimal_path[i]
        ][
            optimal_path[i + 1]
        ]

    print("\nVerification:")
    print("Calculated tour cost:", calculated_cost)

    if calculated_cost == minimum_cost:
        print("✓ Optimal cost verified.")


# ---------------------------------------------------------
# EXECUTE PROGRAM
# ---------------------------------------------------------

if __name__ == "__main__":
    main()


TRAVELLING SALESMAN PROBLEM
BRANCH AND BOUND USING REDUCED COST MATRIX

Cost Matrix:
---------------------------------------------
     INF      20      30      10      11
      15     INF      16       4       2
       3       5     INF       2       4
      19       6      18     INF       3
      16       4       7      16     INF
---------------------------------------------

------------------------------------------------------------
Optimal Tour:
0 -> 3 -> 1 -> 4 -> 2 -> 0

Minimum Tour Cost:
53

Nodes Explored:
21
------------------------------------------------------------

Verification:
Calculated tour cost: 28
